# 🧠 AI Resume Analyzer
> NLP-powered resume ↔ job description matcher with actionable feedback

**Stack:** Python · spaCy · NLTK · scikit-learn · Streamlit

Run each cell in order. The last cell launches the app and gives you a public URL via **ngrok**.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q streamlit spacy nltk scikit-learn plotly pdfplumber python-docx PyPDF2 pyngrok
!python -m spacy download en_core_web_sm -q

import nltk
for pkg in ['stopwords', 'punkt', 'wordnet']:
    nltk.download(pkg, quiet=True)

print('✅ All dependencies installed!')

In [ ]:
# ── Cell 2: Clone the repository ─────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ai-resume-analyzer.git'  # ← update this

if not os.path.exists('ai-resume-analyzer'):
    !git clone $REPO_URL
else:
    print('Repo already cloned.')

%cd ai-resume-analyzer
print('✅ Repository ready!')

In [ ]:
# ── Cell 3: Launch Streamlit via ngrok ───────────────────────────────────────
# Get a free ngrok token at https://ngrok.com (takes 30 seconds)
NGROK_TOKEN = 'YOUR_NGROK_TOKEN'  # ← paste your token here

from pyngrok import ngrok, conf
import subprocess, time, threading

conf.get_default().auth_token = NGROK_TOKEN

# Kill any existing Streamlit processes
!pkill -f streamlit 2>/dev/null; sleep 1

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port=8501',
     '--server.headless=true', '--server.enableCORS=false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

time.sleep(3)

# Open ngrok tunnel
tunnel = ngrok.connect(8501)
print('\n' + '='*55)
print(f'🚀  App is LIVE at: {tunnel.public_url}')
print('='*55)
print('Open the link above in any browser.')
print('Stop: Runtime → Disconnect and delete runtime')

---
## 🔬 Standalone NLP Demo (no UI)
Run this cell to test the analyzer without the web app:

In [ ]:
import sys
sys.path.insert(0, '.')

from utils.analyzer import ResumeAnalyzer

SAMPLE_RESUME = """
John Doe | john@example.com
B.Tech in Computer Science, IIT Delhi (2019)
3 years of experience in Python, Machine Learning, NLP.

Skills: Python, TensorFlow, PyTorch, NLTK, spaCy, SQL, Docker, Git, REST APIs

Experience:
- ML Engineer at Startup X (2021-2024): Built NLP pipelines, deployed models on AWS
- Intern at Company Y (2019): Data analysis with pandas and sklearn
"""

SAMPLE_JD = """
We are looking for an ML Engineer with:
- 2+ years of experience in Python and Machine Learning
- Proficiency in TensorFlow or PyTorch
- Experience with NLP, BERT, transformers
- Knowledge of Docker, Kubernetes, and AWS
- Strong SQL and data engineering skills
- Bachelor's degree in CS or related field
"""

analyzer = ResumeAnalyzer()
result = analyzer.analyze(SAMPLE_RESUME, SAMPLE_JD)

print(f"\n{'='*50}")
print(f"  OVERALL MATCH SCORE: {result['overall_score']}%")
print(f"{'='*50}")
print(f"\n📊 Section Scores:")
for k, v in result['section_scores'].items():
    bar = '█' * (v // 10) + '░' * (10 - v // 10)
    print(f"  {k:<12} [{bar}] {v}%")

print(f"\n✅ Matched Skills ({len(result['matched_skills'])}):\n  {', '.join(result['matched_skills'])}")
print(f"\n❌ Missing Skills ({len(result['missing_skills'])}):\n  {', '.join(result['missing_skills'])}")
print(f"\n🎓 Education: {result['education_match']}")
print(f"💼 Experience: {result['experience_level']}")
print(f"\n💡 Feedback:")
for f in result['feedback']:
    print(f"  • {f}")